# Tarea 4: Análisis de Datos y Optimización

## **Camila Luana Guevara Davila**

**Fecha de entrega:** [VER CANVAS]

**Puntaje total:** 20 puntos

**Instrucciones:**
- Completa los tres problemas en este notebook
- Escribe tu código en las celdas indicadas
- Ejecuta todas las celdas para verificar que tu código funciona
- Guarda tu archivo `.ipynb` en la carpeta `tareas` de tu repositorio privado de GitHub (compartido con el docente)
- Envía el enlace a tu notebook en Canvas

**⚠️ IMPORTANTE:** GitHub registra el historial de cambios de cada archivo. Tu notebook debe ser subido a GitHub **antes del plazo**. **NO** modifiques el archivo después del plazo — los cambios tardíos serán detectados y pueden resultar en penalidad.

**Integridad académica:** Esta es una tarea individual. Puedes consultar los materiales del curso, documentación de Python, herramientas de IA y discutir conceptos con compañeros, pero todo el código debe ser tuyo.

---

In [1]:
# Importaciones estándar - ejecuta esta celda primero
import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import minimize, curve_fit
from io import StringIO

---
## Problema 1: Análisis de Calidad de Agua en Ríos Andino-Amazónicos (7 puntos)

Estás analizando datos de calidad de agua de estaciones de monitoreo en tres ríos de la cuenca amazónica peruana, provenientes de la red de monitoreo de la **Autoridad Nacional del Agua (ANA)**. El conjunto de datos contiene mediciones de temperatura, oxígeno disuelto (OD), pH y conductividad eléctrica recolectadas durante varios meses.

### Tus Tareas:

**Parte A (2 puntos):** Carga y explora los datos
1. Carga los datos del string CSV provisto abajo en un DataFrame de pandas
2. Muestra información básica del conjunto de datos (forma, tipos de datos, primeras filas)
3. Verifica valores faltantes e indica cuántos hay en cada columna
4. Convierte la columna `fecha` a formato datetime usando `pd.to_datetime()`

**Parte B (3 puntos):** Análisis de datos con agrupación
1. Calcula la media, desviación estándar, mínimo y máximo de oxígeno disuelto (`od_mg_l`) agrupado por `estacion_id`
2. Determina qué estación tiene la media más baja de oxígeno disuelto
3. Crea una nueva columna llamada `estado_od` que clasifique cada medición como:
   - "Crítico" si OD < 4 mg/L
   - "Bajo" si OD está entre 4 y 6 mg/L
   - "Adecuado" si OD está entre 6 y 8 mg/L
   - "Bueno" si OD >= 8 mg/L
4. Cuenta cuántas mediciones caen en cada categoría de `estado_od` por estación

**Parte C (2 puntos):** Filtrado y resumen
1. Filtra los datos para incluir solo mediciones donde temperatura > 20°C Y pH entre 6.5 y 8.5
2. Para este subconjunto filtrado, calcula la conductividad media por mes (pista: extrae el mes de la fecha)
3. Identifica qué combinación estación-mes tuvo el mayor número de lecturas con OD "Crítico" o "Bajo"

In [2]:
# Dataset de calidad de agua - ríos andino-amazónicos del Perú
calidad_agua_csv = (
    "estacion_id,fecha,temp_c,od_mg_l,ph,conductividad_us\n"
    "RIO_UCAYALI,2024-05-15,24.3,7.2,7.1,145\n"
    "RIO_UCAYALI,2024-05-22,25.1,6.8,7.0,152\n"
    "RIO_UCAYALI,2024-06-05,24.8,6.5,6.9,158\n"
    "RIO_UCAYALI,2024-06-19,25.5,5.8,6.8,165\n"
    "RIO_UCAYALI,2024-07-03,24.2,5.2,7.0,172\n"
    "RIO_UCAYALI,2024-07-17,23.8,4.8,7.1,168\n"
    "RIO_UCAYALI,2024-08-01,24.5,4.2,7.2,175\n"
    "RIO_UCAYALI,2024-08-15,25.2,5.0,7.0,169\n"
    "RIO_TAMBOPATA,2024-05-15,21.8,8.5,7.4,98\n"
    "RIO_TAMBOPATA,2024-05-22,22.5,8.1,7.5,105\n"
    "RIO_TAMBOPATA,2024-06-05,22.9,7.8,7.3,112\n"
    "RIO_TAMBOPATA,2024-06-19,23.4,7.2,7.2,118\n"
    "RIO_TAMBOPATA,2024-07-03,22.1,6.8,7.1,125\n"
    "RIO_TAMBOPATA,2024-07-17,21.8,6.5,7.0,121\n"
    "RIO_TAMBOPATA,2024-08-01,22.5,6.9,7.1,128\n"
    "RIO_TAMBOPATA,2024-08-15,23.1,7.2,7.2,115\n"
    "RIO_MANTARO,2024-05-15,13.1,9.5,6.5,312\n"
    "RIO_MANTARO,2024-05-22,14.2,8.8,6.4,325\n"
    "RIO_MANTARO,2024-06-05,12.8,8.2,6.3,338\n"
    "RIO_MANTARO,2024-06-19,13.5,7.5,6.2,352\n"
    "RIO_MANTARO,2024-07-03,11.9,6.8,6.0,368\n"
    "RIO_MANTARO,2024-07-17,10.8,5.9,5.9,378\n"
    "RIO_MANTARO,2024-08-01,11.5,5.2,6.1,385\n"
    "RIO_MANTARO,2024-08-15,12.3,4.8,6.2,372\n"
)

# Parte A: Carga y explora los datos
from io import StringIO
# Pista: Usa pd.read_csv(StringIO(calidad_agua_csv))
#Item 1
df = pd.read_csv(StringIO(calidad_agua_csv))
#Item 2
print("Forma del DataFrame:")
print(df.shape)
print("\nTipos de datos:")
print(df.dtypes)
print("\nPrimeras filas:")
print(df.head())
#Item 3
print("\nValores faltantes por columna:")
for i in df.columns:
    cantidad_faltantes = df[i].isnull().sum()
    print(i, ":", cantidad_faltantes)
#Item 4
df["fecha"] = pd.to_datetime(df["fecha"])

print("\nTipos de datos después de convertir la fecha:")
print(df.dtypes)


Forma del DataFrame:
(24, 6)

Tipos de datos:
estacion_id          object
fecha                object
temp_c              float64
od_mg_l             float64
ph                  float64
conductividad_us      int64
dtype: object

Primeras filas:
   estacion_id       fecha  temp_c  od_mg_l   ph  conductividad_us
0  RIO_UCAYALI  2024-05-15    24.3      7.2  7.1               145
1  RIO_UCAYALI  2024-05-22    25.1      6.8  7.0               152
2  RIO_UCAYALI  2024-06-05    24.8      6.5  6.9               158
3  RIO_UCAYALI  2024-06-19    25.5      5.8  6.8               165
4  RIO_UCAYALI  2024-07-03    24.2      5.2  7.0               172

Valores faltantes por columna:
estacion_id : 0
fecha : 0
temp_c : 0
od_mg_l : 0
ph : 0
conductividad_us : 0

Tipos de datos después de convertir la fecha:
estacion_id                 object
fecha               datetime64[ns]
temp_c                     float64
od_mg_l                    float64
ph                         float64
conductividad_us      

In [13]:
# Parte B: Análisis de datos con agrupación
# Item 1
# Calcular las estadísticas del oxígeno disuelto por estación
estadisticas_od = df.groupby("estacion_id")["od_mg_l"].agg(
    ["mean", "std", "min", "max"]
)
print("Estadísticas del oxígeno disuelto por estación:")
print(estadisticas_od)

# Item 2
media_od = df.groupby("estacion_id")["od_mg_l"].mean()
media_od
estacion_media_mas_baja = media_od.idxmin()
valor_media_mas_baja = media_od.min()

print("\nEstación con la media más baja de oxígeno disuelto:")
print(estacion_media_mas_baja)
print("Media de oxígeno disuelto:", valor_media_mas_baja)


# Item 3
def clasificar_od(od):
    if od < 4:
        return "Crítico"
    elif od < 6:
        return "Bajo"
    elif od < 8:
        return "Adecuado"
    else:
        return "Bueno"

# Crear la nueva columna aplicando la función
df["estado_od"] = df["od_mg_l"].apply(clasificar_od)

print("\nDataFrame con la clasificación del oxígeno disuelto:")
print(df)


# Item 4
grupos = df.groupby(["estacion_id", "estado_od"])
conteo_estado_od = grupos.size()

print("\nCantidad de mediciones por categoría y estación:")
print(conteo_estado_od)

Estadísticas del oxígeno disuelto por estación:
                 mean       std  min  max
estacion_id                              
RIO_MANTARO    7.0875  1.709166  4.8  9.5
RIO_TAMBOPATA  7.3750  0.692305  6.5  8.5
RIO_UCAYALI    5.6875  1.062931  4.2  7.2

Estación con la media más baja de oxígeno disuelto:
RIO_UCAYALI
Media de oxígeno disuelto: 5.6875

DataFrame con la clasificación del oxígeno disuelto:
      estacion_id      fecha  temp_c  od_mg_l   ph  conductividad_us estado_od
0     RIO_UCAYALI 2024-05-15    24.3      7.2  7.1               145  Adecuado
1     RIO_UCAYALI 2024-05-22    25.1      6.8  7.0               152  Adecuado
2     RIO_UCAYALI 2024-06-05    24.8      6.5  6.9               158  Adecuado
3     RIO_UCAYALI 2024-06-19    25.5      5.8  6.8               165      Bajo
4     RIO_UCAYALI 2024-07-03    24.2      5.2  7.0               172      Bajo
5     RIO_UCAYALI 2024-07-17    23.8      4.8  7.1               168      Bajo
6     RIO_UCAYALI 2024-08-01    24.5

In [21]:
# Parte C: Filtrado y resumen
# Item 1
# Filtrar temperatura mayor a 20 °C y pH entre 6.5 y 8.5
df_filtrado = df[(df["temp_c"] > 20) & (df["ph"] >= 6.5) & (df["ph"] <= 8.5)]
print("Datos filtrados:")
print(df_filtrado)

# Item 2
# Crear una columna con el mes de cada medición
df_filtrado_2 = df_filtrado.copy()
df_filtrado_2["mes"] = df_filtrado_2["fecha"].dt.month

# Calcular la conductividad media por mes
conductividad_media = df_filtrado_2.groupby("mes")["conductividad_us"].mean()

print("\nConductividad media por mes:")
print(conductividad_media)

# Item 3
df["mes"] = df["fecha"].dt.month

od_critico_bajo = df[(df["estado_od"] == "Crítico") | (df["estado_od"] == "Bajo")]
conteo = od_critico_bajo.groupby(["estacion_id", "mes"]).size()

mayor_cantidad = conteo.max()

print("\nCombinaciones con mayor cantidad de lecturas:")

for combinacion, cantidad in conteo.items():
    if cantidad == mayor_cantidad:
        print(combinacion, ":", cantidad)


Datos filtrados:
      estacion_id      fecha  temp_c  od_mg_l   ph  conductividad_us  \
0     RIO_UCAYALI 2024-05-15    24.3      7.2  7.1               145   
1     RIO_UCAYALI 2024-05-22    25.1      6.8  7.0               152   
2     RIO_UCAYALI 2024-06-05    24.8      6.5  6.9               158   
3     RIO_UCAYALI 2024-06-19    25.5      5.8  6.8               165   
4     RIO_UCAYALI 2024-07-03    24.2      5.2  7.0               172   
5     RIO_UCAYALI 2024-07-17    23.8      4.8  7.1               168   
6     RIO_UCAYALI 2024-08-01    24.5      4.2  7.2               175   
7     RIO_UCAYALI 2024-08-15    25.2      5.0  7.0               169   
8   RIO_TAMBOPATA 2024-05-15    21.8      8.5  7.4                98   
9   RIO_TAMBOPATA 2024-05-22    22.5      8.1  7.5               105   
10  RIO_TAMBOPATA 2024-06-05    22.9      7.8  7.3               112   
11  RIO_TAMBOPATA 2024-06-19    23.4      7.2  7.2               118   
12  RIO_TAMBOPATA 2024-07-03    22.1      6.8  

---
## Problema 2: Comparación Estadística de Parcelas Forestales en la Amazonía (6 puntos)

Investigadores del **INIA (Instituto Nacional de Innovación Agraria) - Estación Experimental Pucallpa** midieron la biomasa arbórea (kg) en parcelas pareadas — unas sometidas a un tratamiento de aprovechamiento forestal de impacto reducido (AFIR) y otras dejadas como control. Se quiere determinar si el tratamiento afectó significativamente la biomasa individual de los árboles y si existe relación entre el diámetro y la biomasa.

### Tus Tareas:

**Parte A (2 puntos):** Comparación de grupos de tratamiento
1. Calcula estadísticas descriptivas (media, desviación estándar, mediana) de biomasa para cada grupo
2. Realiza una prueba t de dos muestras independientes para determinar si hay diferencia significativa en la biomasa media entre parcelas control y AFIR (α = 0.05)
3. Plantea tu hipótesis nula y alternativa, reporta el estadístico t y el p-valor, y escribe una conclusión

**Parte B (2 puntos):** Análisis de correlación
1. Calcula el coeficiente de correlación de Pearson entre el DAP y la biomasa para todo el conjunto de datos
2. Evalúa si esta correlación es estadísticamente significativa (α = 0.05)
3. Interpreta la fuerza y dirección de la correlación

**Parte C (2 puntos):** Ajuste de distribución
1. Ajusta una distribución normal a los datos de biomasa de las parcelas control
2. Reporta los parámetros ajustados (μ y σ)
3. Calcula la probabilidad de que un árbol seleccionado aleatoriamente de las parcelas control tenga biomasa > 150 kg
4. ¿Qué valor de biomasa representa el percentil 90 para los árboles de parcelas control?

In [22]:
# Datos de parcelas forestales - INIA Pucallpa
np.random.seed(458)  # Para reproducibilidad

# Parcelas control: bosque sin intervención
n_control = 35
dap_control = np.random.uniform(15, 50, n_control)  # DAP en cm
biomasa_control = 0.1 * dap_control**2.2 + np.random.normal(0, 15, n_control)
biomasa_control = np.maximum(biomasa_control, 10)  # Asegurar valores positivos

# Parcelas AFIR: los árboles remanentes disponen de más recursos
n_afir = 30
dap_afir = np.random.uniform(18, 55, n_afir)  # DAP en cm
biomasa_afir = 0.12 * dap_afir**2.2 + np.random.normal(5, 18, n_afir)
biomasa_afir = np.maximum(biomasa_afir, 10)

# Crear DataFrame
bosque_df = pd.DataFrame({
    'dap_cm': np.concatenate([dap_control, dap_afir]),
    'biomasa_kg': np.concatenate([biomasa_control, biomasa_afir]),
    'tratamiento': ['Control']*n_control + ['AFIR']*n_afir
})

print(bosque_df.head())
print(f"\nEspecies representativas: Caoba (Swietenia macrophylla), Cedro (Cedrela odorata), Tornillo (Cedrelinga cateniformis)")

      dap_cm  biomasa_kg tratamiento
0  43.208835  395.956074     Control
1  49.475851  521.291644     Control
2  19.647405   54.799539     Control
3  23.651882  119.264060     Control
4  40.431164  335.576313     Control

Especies representativas: Caoba (Swietenia macrophylla), Cedro (Cedrela odorata), Tornillo (Cedrelinga cateniformis)


In [26]:
# Parte A: Comparación de grupos de tratamiento
from scipy import stats

# Item 1
estadisticas_biomasa = bosque_df.groupby("tratamiento")["biomasa_kg"].agg(["mean", "std", "median"])
print("Estadísticas descriptivas de biomasa:")
print(estadisticas_biomasa)

# Item 2
# Separar la biomasa de las parcelas Control y AFIR
grupo_control = bosque_df[bosque_df["tratamiento"] == "Control"]["biomasa_kg"]
grupo_afir = bosque_df[bosque_df["tratamiento"] == "AFIR"]["biomasa_kg"]

# Realizar la prueba t de muestras independientes
t_estadistico, p_valor = stats.ttest_ind(
    grupo_control,
    grupo_afir,
    equal_var=False)

print("\nEstadístico t:", t_estadistico)
print("P-valor:", p_valor)

# Item 3
print()
print('Parte Final')
print("H0: No existe diferencia significativa entre la biomasa media de Control y AFIR.")
print("H1: Existe diferencia significativa entre la biomasa media de Control y AFIR.")

alpha = 0.05

if p_valor < alpha:
    print("Se rechaza H0.")
    print("El tratamiento AFIR produjo una diferencia significativa en la biomasa media.")
else:
    print("No se rechaza H0.")
    print("El tratamiento AFIR no produjo una diferencia significativa en la biomasa media.")


Estadísticas descriptivas de biomasa:
                   mean         std      median
tratamiento                                    
AFIR         325.418953  213.516434  239.121336
Control      269.642640  158.210017  232.870089

Estadístico t: -1.179860839735078
P-valor: 0.24334893995014714

Parte Final
H0: No existe diferencia significativa entre la biomasa media de Control y AFIR.
H1: Existe diferencia significativa entre la biomasa media de Control y AFIR.
No se rechaza H0.
El tratamiento AFIR no produjo una diferencia significativa en la biomasa media.


In [27]:
# Parte B: Análisis de correlación
# Item 1
# Calcular la correlación de Pearson entre DAP y biomasa
correlacion, p_valor_correlacion = stats.pearsonr(
    bosque_df["dap_cm"],
    bosque_df["biomasa_kg"])

print("Coeficiente de correlación:", correlacion)
print("P-valor:", p_valor_correlacion)

# Item 2
alpha = 0.05
if p_valor_correlacion < alpha:
    print("La correlación es estadísticamente significativa.")
else:
    print("La correlación no es estadísticamente significativa.")

# Item 3
# Interpretar la dirección
if correlacion > 0:
    direccion = "positiva"
elif correlacion < 0:
    direccion = "negativa"
else:
    direccion = "nula"

# Interpretar la fuerza
if abs(correlacion) >= 0.7:
    fuerza = "fuerte"
elif abs(correlacion) >= 0.3:
    fuerza = "moderada"
else:
    fuerza = "débil"

print("La correlación es", fuerza, "y", direccion)

Coeficiente de correlación: 0.9564725425194496
P-valor: 2.0975940245401606e-35
La correlación es estadísticamente significativa.
La correlación es fuerte y positiva


In [31]:
# Parte C: Ajuste de distribución
# Item 1 y 2
mu, sigma = stats.norm.fit(grupo_control)
print("Media ajustada (μ):", mu)
print("Desviación estándar ajustada (σ):", sigma)

# Item 3
# Probabilidad de que la biomasa sea mayor a 150 kg
probabilidad = stats.norm.sf(150, loc=mu, scale=sigma)
print("\nProbabilidad de biomasa mayo a 150 kg:", probabilidad * 100, "%")

# Item 4
# Calcular el percentil 90
percentil_90 = stats.norm.ppf(0.90, loc=mu, scale=sigma)

print("\nBiomasa correspondiente al percentil 90:", percentil_90, "kg")


Media ajustada (μ): 269.64263954151227
Desviación estándar ajustada (σ): 155.93349560108524

Probabilidad de biomasa mayo a 150 kg: 77.85386347064977 %

Biomasa correspondiente al percentil 90: 469.47945494992507 kg


---
## Problema 3: Ajuste de Curva de Respuesta a la Luz (7 puntos)

La fotosíntesis depende de la intensidad de luz siguiendo una curva de saturación. La **hipérbola rectangular** se usa comúnmente para modelar esta relación:

$$A = \frac{A_{max} \cdot I}{K + I} - R_d$$

Donde:
- $A$ = tasa de fotosíntesis neta (μmol CO₂ m⁻² s⁻¹)
- $A_{max}$ = tasa máxima de fotosíntesis a saturación de luz
- $I$ = intensidad de luz (μmol fotones m⁻² s⁻¹, PAR)
- $K$ = constante de media saturación (nivel de luz en el que A = A_max/2 - R_d)
- $R_d$ = tasa de respiración en oscuridad (CO₂ liberado cuando I = 0)

Los datos provienen de mediciones de *Cecropia sp.* ("cetico"), una especie pionera característica de la Amazonía peruana, muy importante en la regeneración de bosques perturbados.

### Tus Tareas:

**Parte A (2 puntos):** Define el modelo y la función de costo
1. Escribe una función `respuesta_luz(I, Amax, K, Rd)` que implemente la ecuación anterior
2. Escribe una función de costo `respuesta_luz_mse(params, I_datos, A_datos)` que calcule el error cuadrático medio entre las tasas de fotosíntesis observadas y predichas
3. Prueba tu función `respuesta_luz` calculando A para I = 500 con Amax=25, K=200, Rd=2

**Parte B (3 puntos):** Ajusta el modelo usando optimización
1. Usa `scipy.optimize.minimize` para encontrar los parámetros óptimos (Amax, K, Rd) que minimicen el MSE
2. Usa valores iniciales: Amax=20, K=150, Rd=1
3. Reporta los parámetros ajustados y el MSE final
4. Ajusta también el modelo usando `scipy.optimize.curve_fit` y compara los resultados

**Parte C (2 puntos):** Evalúa e interpreta el modelo
1. Calcula los valores de fotosíntesis predichos usando tus parámetros ajustados
2. Calcula R² (coeficiente de determinación) para evaluar el ajuste del modelo:
   $$R^2 = 1 - \frac{SS_{res}}{SS_{tot}} = 1 - \frac{\sum(y_i - \hat{y}_i)^2}{\sum(y_i - \bar{y})^2}$$
3. Calcula el **punto de compensación lumínico** (el nivel de luz donde A = 0, es decir, la fotosíntesis iguala a la respiración). Pista: despeja I cuando A = 0
4. ¿Cuál es la tasa de fotosíntesis a saturación lumínica (Amax - Rd)?

In [32]:
# Datos de curva de respuesta a la luz - Cecropia sp. (cetico)
# Mediciones en parcela de investigación, Madre de Dios

# PAR (radiación fotosintéticamente activa) en μmol fotones m⁻² s⁻¹
par_datos = np.array([0, 25, 50, 75, 100, 150, 200, 300, 400, 600, 800, 1000, 1200, 1500, 1800])

# Tasa de fotosíntesis neta en μmol CO₂ m⁻² s⁻¹
foto_datos = np.array([-1.8, 1.2, 4.5, 7.1, 9.2, 12.5, 14.8, 17.5, 19.2, 21.1, 22.0, 22.5, 22.8, 23.0, 23.1])

print(f"Rango PAR: {par_datos.min()} a {par_datos.max()} μmol fotones m⁻² s⁻¹")
print(f"Rango fotosíntesis: {foto_datos.min()} a {foto_datos.max()} μmol CO₂ m⁻² s⁻¹")
print("Especie: Cecropia sp. (cetico) - pionera amazónica")

Rango PAR: 0 a 1800 μmol fotones m⁻² s⁻¹
Rango fotosíntesis: -1.8 a 23.1 μmol CO₂ m⁻² s⁻¹
Especie: Cecropia sp. (cetico) - pionera amazónica


In [34]:
# Parte A: Define el modelo y la función de costo

# Item 1
# Función de respuesta a la luz: valores predichos por el modelo
def respuesta_luz(I, Amax, K, Rd):
    A = (Amax * I) / (K + I) - Rd
    return A

# Item 2
# Función de costo: error cuadrático medio
def respuesta_luz_mse(params, I_datos, A_datos):
    Amax = params[0]
    K = params[1]
    Rd = params[2]

    A_predicha = respuesta_luz(I_datos, Amax, K, Rd)

    errores = A_datos - A_predicha
    mse = np.mean(errores**2)
    return mse


# Item 3
# Probar la función con los valores indicados
A_prueba = respuesta_luz(500, 25, 200, 2)
print("Tasa de fotosíntesis para I = 500, Amax=25, K=200, Rd=2:")
print(A_prueba)


Tasa de fotosíntesis para I = 500, Amax=25, K=200, Rd=2:
15.857142857142858


In [41]:
# Parte B: Ajusta el modelo usando optimización

from scipy.optimize import minimize, curve_fit

# Item 1 y 2
parametros_iniciales = [20, 150, 1]   #Amax, K, Rd

# Ajustar el modelo con minimize
resultado = minimize(respuesta_luz_mse,parametros_iniciales,args=(par_datos, foto_datos))

# Guardar los parámetros encontrados
Amax_minimize = resultado.x[0]
K_minimize = resultado.x[1]
Rd_minimize = resultado.x[2]

print('Amax_minimize:', Amax_minimize)
print('K_minimize:', K_minimize)
print('Rd_minimize:', Rd_minimize)

# Item 3
# Calcular el MSE final
mse_minimize = respuesta_luz_mse(
    resultado.x,
    par_datos,
    foto_datos)
print('mse_minimize:', mse_minimize)

#Item 4
resultado_curve_fit = curve_fit(respuesta_luz,par_datos,foto_datos,parametros_iniciales)
parametros_curve_fit = resultado_curve_fit[0]
print("\nParámetros con curve_fit:")
print(parametros_curve_fit)

print('Los resultados obtenidos con minimize y curve_fit son prácticamente iguales, por lo que ambos métodos lograron un ajuste consistente del modelo de respuesta a la luz')


Amax_minimize: 28.44605494901702
K_minimize: 134.75452985437354
Rd_minimize: 2.6271705321314256
mse_minimize: 0.23733661466729108

Parámetros con curve_fit:
[ 28.4460244  134.7545062    2.62715011]
Los resultados obtenidos con minimize y curve_fit son prácticamente iguales, por lo que ambos métodos lograron un ajuste consistente del modelo de respuesta a la luz


In [44]:
# Parte C: Evalúa e interpreta el modelo
# Item 1
foto_predicha = respuesta_luz(par_datos,Amax_minimize,K_minimize,Rd_minimize)
print("Valores de fotosíntesis predichos:")
print(foto_predicha)

# Item 2
# Calcular R cuadrado
ss_res = np.sum((foto_datos - foto_predicha)**2)
media_foto = np.mean(foto_datos)
ss_tot = np.sum((foto_datos - media_foto)**2)
r_cuadrado = 1 - (ss_res / ss_tot)
print("\nR cuadrado:", r_cuadrado)

# Item 3
# Nivel de luz donde A = 0
I_compensacion = (Rd_minimize * K_minimize) / (Amax_minimize - Rd_minimize)
print("\nPunto de compensación lumínico:", I_compensacion)

# Item 4
fotosintesis_saturacion_luminica = Amax_minimize - Rd_minimize
print("\nFotosíntesis a saturación lumínica:", fotosintesis_saturacion_luminica)

Valores de fotosíntesis predichos:
[-2.62717053  1.82435503  5.07116709  7.54402397  9.49019094 12.35734348
 14.368002   17.00187505 18.65067069 20.60185668 21.71809149 22.44085449
 22.94701946 23.47404628 23.8376332 ]

R cuadrado: 0.996574978815181

Punto de compensación lumínico: 13.711790338745345

Fotosíntesis a saturación lumínica: 25.818884416885595


---
## Lista de verificación para entrega

Antes de entregar, verifica que:

- [ ] Todas las celdas de código se ejecutan sin errores
- [ ] Los tres problemas están completos
- [ ] Los resultados son visibles en todas las celdas
- [ ] Tu nombre está incluido al inicio
- [ ] El archivo está guardado en la carpeta `tareas` de tu repositorio privado de GitHub
- [ ] El archivo está subido a GitHub **antes del plazo**
- [ ] El enlace a tu notebook está enviado en Canvas